# SciGraphAgent Benchmark — Notebook 02
## Building Retrieval Systems: Vector Index + Knowledge Graph

---

**Notebook series:**
- Notebook 00 — Setup and foundations ✓
- Notebook 01 — Loading benchmark datasets ✓
- **Notebook 02 — Building retrieval systems ← you are here**
- Notebook 03 — Running experiments (next)
- Notebook 04 — Metrics and results
- Notebook 05 — Visualisation

**What this notebook teaches:**
- What an embedding is — how text becomes numbers
- What a vector database is and how similarity search works
- What a knowledge graph is — nodes, edges, and traversal
- What BFS (Breadth-First Search) is — step by step
- Why we combine vector search AND graph traversal
- What alpha-fusion means (α = 0.6)
- How the four retrieval conditions differ
- What every competing system actually uses for retrieval
- How `step02_build_retrieval_systems.py` works — line by line

---

## Table of Contents

1. Environment setup
2. What is retrieval and why do we need it?
3. What is an embedding? — How text becomes numbers
4. What is a vector database?
5. Building the ChromaDB vector index
6. What is a knowledge graph?
7. Building the knowledge graph
8. What is BFS (Breadth-First Search)?
9. Why combine vector + graph? — The alpha-fusion design
10. What do competing systems actually use?
11. The four retrieval conditions
12. Running `step02_build_retrieval_systems.py` — line by line
13. Comparing all four conditions on real questions
14. Pre-commit verification and commit

---
## Section 1 — Environment Setup

Run this cell first, every time you open this notebook.

In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path

cwd = Path(os.getcwd())
dotenv_path = cwd.parent / ".env" if (cwd.parent / ".env").exists() else cwd / ".env"
load_dotenv(dotenv_path)

groq_key = os.environ.get("GROQ_API_KEY", "")
hf_token  = os.environ.get("HF_TOKEN", "")

print("Working directory:", cwd)
print("GROQ_API_KEY:", groq_key[:8] + "****" if groq_key else "NOT FOUND")
print("HF_TOKEN    :", hf_token[:8]  + "****" if hf_token  else "NOT FOUND")

# Check data files from step01
data_dir = cwd / "data"
if not data_dir.exists():
    data_dir = cwd.parent / "data"

print()
print("Data files available:")
for f in sorted(data_dir.glob("*_sample_*.json")):
    print(f"  ✓ {f.name}")

print("\n✓ Environment ready")

Working directory: /run/media/bala/HDD/Projects/scigraphagent-benchmark
GROQ_API_KEY: gsk_GWHq****
HF_TOKEN    : hf_kkSXQ****

Data files available:
  ✓ 2wikimultihopqa_sample_3.json
  ✓ hotpotqa_sample_3.json
  ✓ musique_sample_3.json

✓ Environment ready


---
## Section 2 — What is Retrieval and Why Do We Need It?

### The memory problem with LLMs

A Large Language Model learns from billions of documents during training. After training, its knowledge is **frozen** — it cannot learn new facts. If a paper was published after the training cutoff, the model does not know about it.

More critically, even for facts the model learned during training, it can **hallucinate** — confidently state something that is wrong because it is pattern-matching on text rather than looking up a verified fact.

### The retrieval solution

Instead of asking the model to remember, we:
1. **Store** documents in a database
2. **Retrieve** the relevant ones at query time
3. **Show** them to the model as context: "Here is the relevant information. Answer based only on this."

This is Retrieval-Augmented Generation (RAG). The model's job becomes simpler — it reads provided text and extracts an answer — rather than the harder job of remembering facts from training.

### The retrieval challenge

Given a question and a database of thousands of passages, how do you find the **right** passages quickly?

You cannot read all of them — that would be too slow. You need a way to find relevant passages without reading every single one. This is the retrieval problem, and it has two main solutions:

1. **Vector similarity search** — convert text to numbers, find the numerically closest ones
2. **Graph traversal** — follow connections between entities to find related passages

We use **both**, and this notebook builds both.

---
## Section 3 — What is an Embedding? How Text Becomes Numbers

### The core idea

Computers cannot directly compare text strings for meaning. "Dog" and "canine" are completely different strings, but they mean the same thing. "Bank" (financial institution) and "bank" (river edge) are identical strings but mean different things.

An **embedding** converts a piece of text into a list of numbers (a vector) such that:
- Texts with **similar meaning** produce vectors that are **close together** in number-space
- Texts with **different meaning** produce vectors that are **far apart**

### The 384-dimensional space

Our embedding model (`all-MiniLM-L6-v2`) converts any text into exactly 384 numbers. Every piece of text — whether one word or one paragraph — becomes a list of exactly 384 floating-point numbers.

This 384-number list is the embedding. The 384 numbers together represent the "meaning" of the text.

### Why 384 dimensions specifically?

This is a deliberate engineering trade-off:

| Model | Dimensions | Quality | RAM per 1M chunks | Speed on CPU |
|---|---|---|---|---|
| all-MiniLM-L6-v2 (ours) | 384 | Good | ~1.5 GB | ~10ms/chunk |
| nomic-embed-text-v1.5 (planned) | 768 | Better | ~3 GB | ~25ms/chunk |
| text-embedding-ada-002 (OpenAI) | 1536 | Best | ~6 GB | Cloud only |

384 dimensions fits thousands of chunks in our 8 GB RAM machine, runs entirely on CPU, and is fast enough for the benchmark. The planned upgrade to `nomic-embed-text-v1.5` (768-dim) is used by GraphAgents (Stewart et al. 2026) and performs better on scientific vocabulary.

### Cosine similarity — how we compare embeddings

Given two 384-dimensional vectors, cosine similarity measures the angle between them:
- **Similarity = 1.0** → identical direction → same meaning
- **Similarity = 0.0** → perpendicular → unrelated
- **Similarity = -1.0** → opposite direction → opposite meaning

In practice, most text pairs score between 0.3 and 0.9.

In [14]:
# Demonstrate embeddings — show how similar texts produce similar vectors
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model (uses cache after first download)...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {model.get_embedding_dimension()}")
print()

# Embed several sentences
sentences = [
    "Oliver Reed was an English actor.",
    "Reed was a British film star born in 1938.",         # semantically similar
    "Knowledge graphs store entities and relationships.", # unrelated
    "The dog chased the cat up the tree.",                # very different
]

embeddings = model.encode(sentences)
print(f"Shape of embeddings array: {embeddings.shape}")
print(f"  → {len(sentences)} sentences × {embeddings.shape[1]} dimensions")
print()

# Show what one embedding looks like (just first 8 numbers)
print(f"First sentence embedding (first 8 of 384 numbers):")
print(f"  {embeddings[0][:8].tolist()}")
print(f"  ... (376 more numbers)")
print()

# Compute cosine similarity between all pairs
def cosine_sim(a, b):
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Cosine similarity matrix:")
print(f"{'':>45}", end="")
for j in range(len(sentences)):
    print(f"  S{j+1}", end="")
print()

for i, (sent_i, emb_i) in enumerate(zip(sentences, embeddings)):
    label = sent_i[:42] + "..."
    print(f"S{i+1}: {label:<44}", end="")
    for j, emb_j in enumerate(embeddings):
        sim = cosine_sim(emb_i, emb_j)
        print(f"  {sim:.2f}", end="")
    print()

print()
print("Key observations:")
sim_01 = cosine_sim(embeddings[0], embeddings[1])
sim_02 = cosine_sim(embeddings[0], embeddings[2])
sim_03 = cosine_sim(embeddings[0], embeddings[3])
print(f"  S1 vs S2 (same topic, diff words) : {sim_01:.3f}  ← HIGH (similar meaning)")
print(f"  S1 vs S3 (different topic)        : {sim_02:.3f}  ← LOW  (unrelated)")
print(f"  S1 vs S4 (very different)          : {sim_03:.3f}  ← LOW  (very different)")

Loading embedding model (uses cache after first download)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384

Shape of embeddings array: (4, 384)
  → 4 sentences × 384 dimensions

First sentence embedding (first 8 of 384 numbers):
  [-0.006268886383622885, -0.046172771602869034, -0.08650518208742142, -0.0429827980697155, -0.02026362158358097, 0.07833819091320038, 0.0392608717083931, 0.008189260959625244]
  ... (376 more numbers)

Cosine similarity matrix:
                                               S1  S2  S3  S4
S1: Oliver Reed was an English actor....          1.00  0.69  0.00  0.02
S2: Reed was a British film star born in 1938....  0.69  1.00  0.02  0.00
S3: Knowledge graphs store entities and relati...  0.00  0.02  1.00  -0.02
S4: The dog chased the cat up the tree....        0.02  0.00  -0.02  1.00

Key observations:
  S1 vs S2 (same topic, diff words) : 0.690  ← HIGH (similar meaning)
  S1 vs S3 (different topic)        : 0.003  ← LOW  (unrelated)
  S1 vs S4 (very different)          : 0.015  ← LOW  (very different)


---
## Section 4 — What is a Vector Database?

### The scaling problem

If you have 10,000 text chunks, and a new query arrives, you could compute cosine similarity between the query embedding and all 10,000 chunk embeddings — then return the top 5. This works but takes O(n) time — it gets slower as n grows.

A vector database solves this with an **Approximate Nearest Neighbour (ANN)** index — a data structure that finds the closest vectors in O(log n) time instead of O(n). It is much faster at the cost of occasionally missing the absolute closest vector (hence "approximate").

### ChromaDB — our vector database

ChromaDB is an open-source, pip-installable vector database. It:
- Stores text chunks alongside their 384-dim embeddings
- Builds an HNSW (Hierarchical Navigable Small World) index for fast ANN search
- Persists to disk so you do not need to re-embed on every run
- Returns results sorted by cosine similarity with metadata

### HNSW — how fast nearest-neighbour search works

HNSW builds a multi-layer graph of vectors. The top layer has only a few nodes spread across the space. Each lower layer adds more nodes. At search time:
1. Start at a node in the top layer
2. Greedily move to the neighbour closest to the query
3. Move down to a lower layer
4. Repeat until the bottom layer

This finds approximate nearest neighbours in milliseconds even with millions of vectors.

### Comparison of vector databases

| Database | Open source | Persistent | GPU support | Best for |
|---|---|---|---|---|
| ChromaDB (ours) | ✓ | ✓ disk | ✗ | Dev/research, pip-installable |
| Pinecone | ✗ (cloud) | ✓ cloud | ✓ | Production cloud |
| Weaviate | ✓ | ✓ | ✓ | Production self-hosted |
| FAISS (Facebook) | ✓ | ✗ (memory) | ✓ | Research, large-scale |
| Qdrant | ✓ | ✓ | ✓ | Production, Rust-based |

We use ChromaDB because it is free, pip-installable, persistent, and requires no GPU — matching all our constraints.

In [3]:
# Demonstrate ChromaDB from scratch — understand every operation
import chromadb
from chromadb.utils import embedding_functions
from pathlib import Path

print("=" * 55)
print("CHROMADB DEMONSTRATION")
print("=" * 55)

# 1. Create a client — this is the database connection
#    PersistentClient saves data to disk (survives program restarts)
#    EphemeralClient keeps data in memory only
demo_path = Path("index") / "chroma_demo"
client = chromadb.PersistentClient(path=str(demo_path))
print(f"\n1. Created ChromaDB client at: {demo_path}")

# 2. Define embedding function — how text becomes vectors
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("2. Embedding function: all-MiniLM-L6-v2 (384-dim)")

# 3. Create a collection — like a table in a database
#    metadata={"hnsw:space": "cosine"} means use cosine distance
try:
    client.delete_collection("demo_collection")
except Exception:
    pass
collection = client.create_collection(
    name="demo_collection",
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)
print("3. Created collection 'demo_collection'")

# 4. Add documents — ChromaDB embeds them automatically
documents = [
    "Oliver Reed was an English actor known for his intense screen presence.",
    "The character Bismarck was played by Oliver Reed in Royal Flash.",
    "Royal Flash is a 1975 British comedy film directed by Richard Lester.",
    "Knowledge graphs represent entities and their relationships as nodes and edges.",
    "BFS traversal explores a graph level by level from a starting node.",
]
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "demo", "index": i} for i in range(len(documents))]
)
print(f"4. Added {len(documents)} documents (ChromaDB embedded them automatically)")
print(f"   Collection now has {collection.count()} documents")

# 5. Query — find most similar documents to a query
query = "What nationality was Oliver Reed's character?"
print(f"\n5. Query: '{query}'")

results = collection.query(
    query_texts=[query],      # ChromaDB embeds the query too
    n_results=3,              # return top 3 most similar
    include=["documents", "metadatas", "distances"]
)

print("\nTop 3 results (by cosine similarity):")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    # ChromaDB returns distance (not similarity)
    # Cosine similarity = 1 - cosine distance
    similarity = round(1 - dist, 3)
    print(f"  [{i+1}] similarity={similarity} | {doc[:70]}...")
    print(f"       doc_id=doc_{meta['index']}")

CHROMADB DEMONSTRATION

1. Created ChromaDB client at: index/chroma_demo


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2. Embedding function: all-MiniLM-L6-v2 (384-dim)
3. Created collection 'demo_collection'
4. Added 5 documents (ChromaDB embedded them automatically)
   Collection now has 5 documents

5. Query: 'What nationality was Oliver Reed's character?'

Top 3 results (by cosine similarity):
  [1] similarity=0.783 | Oliver Reed was an English actor known for his intense screen presence...
       doc_id=doc_0
  [2] similarity=0.524 | The character Bismarck was played by Oliver Reed in Royal Flash....
       doc_id=doc_1
  [3] similarity=0.116 | Royal Flash is a 1975 British comedy film directed by Richard Lester....
       doc_id=doc_2


---
## Section 5 — Building the ChromaDB Vector Index

### What `build_vector_index()` does in `step02`

The function takes all downloaded records and:
1. Splits each context passage into sentence-level chunks
2. Assigns each chunk a unique ID: `{record_id}_{chunk_index}`
3. Stores chunk text + metadata (record_id, dataset, answer, q_type) in ChromaDB
4. ChromaDB automatically embeds each chunk with all-MiniLM-L6-v2

### Why chunk at sentence level?

The full SciGraphAgent pipeline uses `RecursiveCharacterTextSplitter` at 512 tokens with 64-token overlap. Here we use sentence splits for simplicity — it demonstrates the same retrieval principle without the LangChain dependency.

Chunking matters because:
- If chunks are too large, one embedding must represent too many different ideas — the embedding becomes an average that matches nothing well
- If chunks are too small, each chunk lacks enough context for the LLM to answer from
- 512 tokens (~350 words) is the empirically validated sweet spot for most RAG tasks

In [4]:
# Build the real vector index from our downloaded data
# This is the same function as in step02_build_retrieval_systems.py

import json, re, time
import chromadb
from chromadb.utils import embedding_functions
from pathlib import Path

DATA_DIR  = Path("data")
INDEX_DIR = Path("index")
INDEX_DIR.mkdir(exist_ok=True)

# Load hotpotqa sample
dataset = "hotpotqa"
n = 3
data_file = DATA_DIR / f"{dataset}_sample_{n}.json"
with open(data_file) as f:
    records = json.load(f)
print(f"Loaded {len(records)} records from {data_file.name}")

# Build ChromaDB client
client = chromadb.PersistentClient(path=str(INDEX_DIR / "chroma"))
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection_name = f"nb02_{dataset}_{n}"
try:
    client.delete_collection(collection_name)
except Exception:
    pass
collection = client.create_collection(
    name=collection_name,
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)

# Chunk and index
documents, metadatas, ids = [], [], []
for rec in records:
    sentences = [
        s.strip()
        for s in re.split(r'(?<=[.!?])\s+', rec["context"])
        if len(s.strip()) > 30
    ]
    for i, sent in enumerate(sentences[:12]):
        documents.append(sent)
        metadatas.append({
            "record_id": rec["id"],
            "answer":    rec["answer"][:60],
            "q_type":    rec["q_type"],
            "chunk_idx": i,
        })
        ids.append(f"{rec['id']}_{i}")

print(f"\nChunking breakdown:")
print(f"  Records       : {len(records)}")
print(f"  Total chunks  : {len(documents)}")
print(f"  Avg per record: {len(documents)/len(records):.1f}")
print(f"\nExample chunks from record 1:")
for i, (doc, meta) in enumerate(zip(documents[:4], metadatas[:4])):
    print(f"  Chunk {i}: '{doc[:70]}...'")

print(f"\nIndexing {len(documents)} chunks into ChromaDB...")
t0 = time.perf_counter()
collection.add(documents=documents, metadatas=metadatas, ids=ids)
elapsed = time.perf_counter() - t0
print(f"Done in {elapsed:.1f}s — uses cache after first run")
print(f"Collection count: {collection.count()} chunks")

Loaded 3 records from hotpotqa_sample_3.json

Chunking breakdown:
  Records       : 3
  Total chunks  : 36
  Avg per record: 12.0

Example chunks from record 1:
  Chunk 0: '[Robin Barton] Robin Barton (born 5 November 1958) is a British art de...'
  Chunk 1: 'Barton studied photography and graphic design at the Exeter College of...'
  Chunk 2: 'Moving to London in 1980 he began working as a freelance photographer ...'
  Chunk 3: 'Laterly he worked for other publications "Sunday Times", "Sunday Teleg...'

Indexing 36 chunks into ChromaDB...
Done in 2.1s — uses cache after first run
Collection count: 36 chunks


---
## Section 6 — What is a Knowledge Graph?

### The limits of vector similarity

Vector similarity finds passages that are *semantically similar* to the query. But for multi-hop questions, this is not enough.

Consider the question: *"What nationality was Oliver Reed's character in Royal Flash?"*

The answer requires:
- Hop 1: Oliver Reed's character in Royal Flash was Bismarck
- Hop 2: Bismarck was Prussian

No single passage says "Oliver Reed's character in Royal Flash was Prussian." The information is spread across two passages that must be *connected*.

### What is a knowledge graph?

A knowledge graph is a network where:
- **Nodes** = entities (people, places, concepts, methods, datasets)
- **Edges** = relationships between entities

```
Oliver Reed ──APPEARED_IN──▶ Royal Flash ──DIRECTED_BY──▶ Richard Lester
     │                            │
PLAYED_ROLE                 SET_IN_PERIOD
     │                            │
     ▼                            ▼
  Bismarck ──NATIONALITY──▶  Prussian        Victorian England
```

Once this graph exists, the multi-hop question can be answered by traversal: start at "Oliver Reed", follow APPEARED_IN to "Royal Flash", follow PLAYED_ROLE to "Bismarck", follow NATIONALITY to "Prussian."

### In the full SciGraphAgent pipeline

The graph is built with typed relations using:
- **Stage 1:** scispaCy NER — identifies 12 scientific entity types (GENE, CHEMICAL, METHOD, DATASET, METRIC, etc.)
- **Stage 2:** Claude relation extraction — classifies pairs into 12 typed relations (USES_METHOD, PROPOSES, EVALUATES, IMPROVES, etc.)

### In this benchmark demonstration

We use **entity co-occurrence** — if two capitalised noun phrases appear near each other in the same passage, we draw an edge between them labelled CO_OCCURS_WITH. This is fast (regex-only, no API calls, no GPU) and demonstrates the same traversal principle.

### Knowledge graph vs relational database

A relational database (SQL) stores data in fixed tables with fixed columns. If you want to add a new type of relationship, you need to add a new table.

A knowledge graph is flexible — you add any entity and any relationship type without changing the schema. This makes it ideal for scientific literature where new concepts and relationships emerge constantly.

In [5]:
# Build a tiny knowledge graph from scratch to understand the structure
from collections import defaultdict

class TinyKnowledgeGraph:
    """
    Minimal knowledge graph for demonstration.
    Nodes: entities (strings)
    Edges: (source, relation, target) triples
    """
    def __init__(self):
        self.triples = []           # list of (source, relation, target)
        self.adj = defaultdict(list) # adjacency: source -> [(relation, target)]

    def add_triple(self, source, relation, target):
        self.triples.append((source, relation, target))
        self.adj[source].append((relation, target))

    def query(self, entity):
        """Return all edges from a given entity."""
        return self.adj.get(entity, [])

    def traverse(self, start, max_hops=2):
        """Return all entities reachable from start within max_hops."""
        visited = {start}
        frontier = [start]
        paths = []
        for hop in range(max_hops):
            next_frontier = []
            for entity in frontier:
                for relation, target in self.adj.get(entity, []):
                    paths.append(f"{entity} --{relation}--> {target}")
                    if target not in visited:
                        visited.add(target)
                        next_frontier.append(target)
            frontier = next_frontier
        return paths

# Build a small graph about the Oliver Reed / Royal Flash question
kg = TinyKnowledgeGraph()
kg.add_triple("Oliver Reed",  "APPEARED_IN",    "Royal Flash")
kg.add_triple("Oliver Reed",  "PLAYED_ROLE",    "Bismarck")
kg.add_triple("Oliver Reed",  "NATIONALITY",    "English")
kg.add_triple("Royal Flash",  "DIRECTED_BY",    "Richard Lester")
kg.add_triple("Royal Flash",  "RELEASED_IN",    "1975")
kg.add_triple("Bismarck",     "NATIONALITY",    "Prussian")
kg.add_triple("Bismarck",     "ROLE_TYPE",      "Antagonist")
kg.add_triple("Richard Lester","ALSO_DIRECTED", "The Three Musketeers")

print("=" * 55)
print("KNOWLEDGE GRAPH DEMONSTRATION")
print("=" * 55)
print(f"\nTotal triples: {len(kg.triples)}")
print("\nAll triples:")
for s, r, t in kg.triples:
    print(f"  ({s}) --{r}--> ({t})")

print()
print("Query: what do we know about 'Oliver Reed'?")
for rel, target in kg.query("Oliver Reed"):
    print(f"  Oliver Reed --{rel}--> {target}")

print()
print("Multi-hop traversal from 'Oliver Reed' (max 2 hops):")
paths = kg.traverse("Oliver Reed", max_hops=2)
for p in paths:
    print(f"  {p}")

print()
print("Answer to 'What nationality was Oliver Reed's character?'")
print("  Hop 1: Oliver Reed --PLAYED_ROLE--> Bismarck")
print("  Hop 2: Bismarck --NATIONALITY--> Prussian")
print("  ✓ Answer: Prussian")

KNOWLEDGE GRAPH DEMONSTRATION

Total triples: 8

All triples:
  (Oliver Reed) --APPEARED_IN--> (Royal Flash)
  (Oliver Reed) --PLAYED_ROLE--> (Bismarck)
  (Oliver Reed) --NATIONALITY--> (English)
  (Royal Flash) --DIRECTED_BY--> (Richard Lester)
  (Royal Flash) --RELEASED_IN--> (1975)
  (Bismarck) --NATIONALITY--> (Prussian)
  (Bismarck) --ROLE_TYPE--> (Antagonist)
  (Richard Lester) --ALSO_DIRECTED--> (The Three Musketeers)

Query: what do we know about 'Oliver Reed'?
  Oliver Reed --APPEARED_IN--> Royal Flash
  Oliver Reed --PLAYED_ROLE--> Bismarck
  Oliver Reed --NATIONALITY--> English

Multi-hop traversal from 'Oliver Reed' (max 2 hops):
  Oliver Reed --APPEARED_IN--> Royal Flash
  Oliver Reed --PLAYED_ROLE--> Bismarck
  Oliver Reed --NATIONALITY--> English
  Royal Flash --DIRECTED_BY--> Richard Lester
  Royal Flash --RELEASED_IN--> 1975
  Bismarck --NATIONALITY--> Prussian
  Bismarck --ROLE_TYPE--> Antagonist

Answer to 'What nationality was Oliver Reed's character?'
  Hop 1: Oliv

---
## Section 7 — Building the Knowledge Graph from Real Data

### Entity co-occurrence extraction

Our benchmark knowledge graph is built from **capitalized noun phrase co-occurrence**. The logic:

1. Find all capitalized phrases in a passage (e.g., "Oliver Reed", "Royal Flash", "Richard Lester")
2. For each adjacent pair of capitalized phrases in the text, add an edge
3. The edge type is always `CO_OCCURS_WITH` — we do not classify the relationship type

This is a significant simplification from the full pipeline (scispaCy + Claude relation extraction), but it demonstrates the same BFS traversal principle.

### Why co-occurrence works well enough for the benchmark

If two entities co-occur repeatedly in the same passages, they are likely related — even without knowing the specific relationship type. For multi-hop question answering, finding that "Oliver Reed" and "Bismarck" co-occur in the same passage is enough to seed the BFS traversal.

In [6]:
# Build the real knowledge graph from downloaded data
import re
from collections import defaultdict

STOPWORDS = {
    "The", "A", "An", "In", "On", "For", "Of", "To",
    "Is", "It", "We", "This", "That", "He", "She",
    "They", "His", "Her", "Its", "At", "By", "As",
}

def extract_entity_pairs(text):
    """Extract capitalised noun phrases and return adjacent pairs."""
    pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})\b'
    entities = re.findall(pattern, text)
    entities = [
        e for e in dict.fromkeys(entities)
        if e not in STOPWORDS
    ]
    return [(entities[i], entities[i+1]) for i in range(len(entities)-1)]

# Build adjacency graph
adj = defaultdict(list)   # entity -> [(neighbour, record_id)]
node_set = set()
edge_count = 0

for rec in records:
    pairs = extract_entity_pairs(rec["context"])
    for a, b in pairs:
        a_norm = a.lower().strip()
        b_norm = b.lower().strip()
        if a_norm == b_norm or len(a_norm) < 3 or len(b_norm) < 3:
            continue
        adj[a_norm].append((b_norm, rec["id"]))
        adj[b_norm].append((a_norm, rec["id"]))
        node_set.update([a_norm, b_norm])
        edge_count += 1

print("Knowledge graph built from co-occurrence extraction")
print(f"  Nodes      : {len(node_set):,}")
print(f"  Edges      : {edge_count:,}")
print(f"  Avg degree : {edge_count/max(len(node_set),1):.2f} edges per node")
print()

# Show the most connected nodes (high degree = important entities)
degree = {node: len(neighbours) for node, neighbours in adj.items()}
top_nodes = sorted(degree.items(), key=lambda x: -x[1])[:10]
print("Top 10 most connected entities:")
for node, deg in top_nodes:
    neighbours = [n for n, _ in adj[node][:3]]
    print(f"  '{node:<25}' degree={deg:3d}  connects to: {neighbours[:3]}")

print()
# Show what the graph knows about Oliver Reed
print("Graph neighbours of 'oliver reed':")
for neighbour, source_id in adj.get("oliver reed", [])[:8]:
    print(f"  oliver reed → CO_OCCURS_WITH → {neighbour}  [from: {source_id[:20]}]")

Knowledge graph built from co-occurrence extraction
  Nodes      : 411
  Edges      : 426
  Avg degree : 1.04 edges per node

Top 10 most connected entities:
  'november                 ' degree=  4  connects to: ['robin barton', 'british', 'singles']
  'london                   ' degree=  4  connects to: ['moving', 'sounds', 'radha krishna temple']
  'american                 ' degree=  4  connects to: ['funny bones', 'hollywood pictures', 'destiny']
  'england                  ' degree=  4  connects to: ['blackpool', 'oliver platt', 'jerusalem']
  'february                 ' degree=  4  connects to: ['robert oliver reed', 'may', 'doll domination']
  'may                      ' degree=  4  connects to: ['february', 'english', 'tour']
  'english                  ' degree=  4  connects to: ['may', 'notable', 'der widerspenstigen']
  'released                 ' degree=  4  connects to: ['colonel muammar gaddafi', 'italy', 'allman brothers band']
  'italy                    ' degree=  4  

---
## Section 8 — What is BFS (Breadth-First Search)?

### Graph traversal — why we need it

Once we have a knowledge graph, we need to **traverse** it to find entities connected to a query. There are two classic traversal strategies:

- **DFS (Depth-First Search):** go as deep as possible down one path before backtracking. Like exploring a maze by always taking the first unexplored turn.
- **BFS (Breadth-First Search):** explore all neighbours at the current depth before going deeper. Like spreading ripples from a stone dropped in water.

We use **BFS** because:
- It finds the shortest path between entities (minimum hops)
- It guarantees we explore all depth-1 neighbours before any depth-2 neighbours
- It naturally stops at a configurable depth (depth=2 in our system)
- It is the approach used by HippoRAG (Gutiérrez et al., 2024) and GraphAgents (Stewart et al., 2026)

### BFS step by step

Starting from seed entity `"oliver reed"` with depth=2:

```
Depth 0:  [oliver reed]                      ← starting point
Depth 1:  [the trap, women, gladiator, ...]  ← all direct neighbours
Depth 2:  [neighbours of depth-1 nodes]      ← two hops away
STOP.
```

### Planned upgrade: Semantic-Stop BFS

GraphAgents (Stewart et al., 2026) introduces **Semantic-Stop BFS** — BFS that only accepts paths passing through a semantically meaningful waypoint node. For example, searching for connections between "PFAS chemicals" and "biomedical tubing" through the waypoint "biocompatible" makes the cross-domain discovery structured and purposeful. This is a planned improvement for SciGraphAgent.

In [7]:
# Implement and demonstrate BFS step by step
from collections import deque

def bfs_step_by_step(adj, seeds, depth=2, max_paths=10):
    """
    BFS traversal with detailed step-by-step output.
    adj   : adjacency dict {node: [(neighbour, record_id)]}
    seeds : starting nodes
    depth : maximum hops
    """
    paths = []
    visited = set()

    # Queue items: (current_node, path_so_far, current_depth)
    queue = deque()
    for seed in seeds:
        if seed in adj:
            queue.append((seed, [seed], 0))
            visited.add(seed)

    print(f"Starting BFS from seeds: {seeds}")
    print(f"Max depth: {depth}")
    print()

    step = 0
    while queue and len(paths) < max_paths:
        current, path_so_far, current_depth = queue.popleft()

        if current_depth >= depth:
            continue

        neighbours = adj.get(current, [])
        print(f"  Step {step+1}: At '{current}' (depth {current_depth}) "
              f"→ {len(neighbours)} neighbours")

        for neighbour, source_id in neighbours[:3]:  # limit for demo
            if neighbour not in visited:
                visited.add(neighbour)
                new_path = path_so_far + [neighbour]
                chain = " → ".join(new_path)
                paths.append(f"{chain}  [source: {source_id[:20]}]")
                print(f"    Found path: {chain}")
                queue.append((neighbour, new_path, current_depth + 1))
        step += 1

    return paths

# Run BFS on the real graph
print("=" * 55)
print("BFS TRAVERSAL DEMONSTRATION")
print("=" * 55)
print()

seeds = ["oliver reed", "royal flash"]
# Filter to seeds that exist in graph
valid_seeds = [s for s in seeds if s in adj]
print(f"Requested seeds: {seeds}")
print(f"Valid seeds (in graph): {valid_seeds}")
print()

paths = bfs_step_by_step(adj, valid_seeds, depth=2, max_paths=8)

print()
print("All paths found:")
for p in paths:
    print(f"  {p}")

BFS TRAVERSAL DEMONSTRATION

Requested seeds: ['oliver reed', 'royal flash']
Valid seeds (in graph): ['oliver reed', 'royal flash']

Starting BFS from seeds: ['oliver reed', 'royal flash']
Max depth: 2

  Step 1: At 'oliver reed' (depth 0) → 2 neighbours
    Found path: oliver reed → sir alec guinness
    Found path: oliver reed → johnny depp
  Step 2: At 'royal flash' (depth 0) → 2 neighbours
    Found path: royal flash → richard lester
    Found path: royal flash → additionally
  Step 3: At 'sir alec guinness' (depth 1) → 2 neighbours
    Found path: oliver reed → sir alec guinness → independent magazine
  Step 4: At 'johnny depp' (depth 1) → 2 neighbours
    Found path: oliver reed → johnny depp → lou reed
  Step 5: At 'richard lester' (depth 1) → 2 neighbours
    Found path: royal flash → richard lester → malcolm
  Step 6: At 'additionally' (depth 1) → 2 neighbours
    Found path: royal flash → additionally → otto

All paths found:
  oliver reed → sir alec guinness  [source: 5add1d

---
## Section 9 — Why Combine Vector + Graph? The Alpha-Fusion Design

### The complementarity finding

Han et al. (2025) — *RAG vs. GraphRAG: A Systematic Evaluation* (arXiv:2502.11371) — is the key paper motivating our design. Their finding:

| Query type | Vector RAG | Graph RAG | Best approach |
|---|---|---|---|
| Single-hop factual | ✓ Strong | ✗ Weak | Vector |
| Multi-hop relational | ✗ Weak | ✓ Strong | Graph |
| Global/thematic | ✗ Weak | ✓ Strong | Graph |
| **Combined** | — | — | **Best overall** |

The two approaches are **complementary, not competitive**. Combining both outperforms either alone across all query types.

### The alpha parameter

We combine the two signals with a weight parameter α (alpha):

```
context = (1 - α) × vector_chunks  +  α × graph_paths
```

- **α = 0.0** → pure vector retrieval (Condition B)
- **α = 1.0** → pure graph retrieval (Condition C)
- **α = 0.6** → hybrid, graph-leaning (Condition D — our proposed system)

Why 0.6 (graph-leaning)? Our benchmark uses multi-hop questions where graph retrieval has a structural advantage. A graph-leaning alpha reflects this.

### Planned improvement: Adaptive alpha

α = 0.6 is a fixed heuristic. The planned improvement (motivated by Han et al. 2025) is an **adaptive alpha** — a query classifier that predicts the optimal α based on the question type:
- Detects single-hop questions → uses α = 0.2 (vector-leaning)
- Detects multi-hop questions → uses α = 0.7 (graph-leaning)

### What GraphRAG-R1 found

GraphRAG-R1 (Yu et al., WWW 2026, arXiv:2507.23581) explicitly names this "hybrid graph-textual retrieval" and shows in their ablation:
- Text+Graph outperforms Text only
- Text+Graph outperforms Graph only
- This validates our alpha-fusion design from an independent published source

In [15]:
# Demonstrate how different alpha values change the context assembly

def assemble_context(query, collection, adj, alpha=0.6, top_k=4):
    """
    Assemble context using alpha-weighted fusion.
    Returns: (context_string, vector_count, graph_count)
    """
    lines = [f"[Context | alpha={alpha}]"]

    # ── Vector branch ─────────────────────────────────────────────
    vector_k = max(1, int(top_k * (1 - alpha)))
    if vector_k > 0 and collection.count() > 0:
        v_results = collection.query(
            query_texts=[query],
            n_results=min(vector_k, collection.count()),
            include=["documents", "distances"]
        )
        lines.append(f"\n-- Vector ({vector_k} chunks, weight {1-alpha:.1f}) --")
        for doc, dist in zip(
            v_results["documents"][0],
            v_results["distances"][0]
        ):
            sim = round(1 - dist, 3)
            lines.append(f"  [sim={sim}] {doc[:60]}...")
        vector_count = len(v_results["documents"][0])
    else:
        vector_count = 0

    # ── Graph branch ──────────────────────────────────────────────
    graph_k = int(top_k * alpha)   # 0 when alpha=0.0, no floor
    if graph_k == 0:
        lines.append("\n-- Graph BFS (disabled, alpha=0.0) --")
        return "\n".join(lines), vector_count, 0
    words = re.findall(r'\b[a-z]{4,}\b', query.lower())
    seeds = [w for w in words if w in adj][:4]
    caps  = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\b', query)
    seeds += [c.lower() for c in caps if c.lower() in adj]
    seeds = list(dict.fromkeys(seeds))[:4]

    paths = []
    visited = set()
    queue = [(s, [s], 0) for s in seeds]
    while queue and len(paths) < graph_k * 3:
        curr, path, d = queue.pop(0)
        if curr in visited or d > 2:
            continue
        visited.add(curr)
        for nb, src in adj.get(curr, [])[:3]:
            if nb not in visited:
                chain = " → ".join(path + [nb])
                paths.append(f"  {chain}")
                queue.append((nb, path + [nb], d + 1))

    lines.append(f"\n-- Graph BFS ({graph_k*3} paths, weight {alpha:.1f}) --")
    lines.append(f"  Seeds: {seeds}")
    lines.extend(paths[:graph_k * 3])
    graph_count = len(paths)

    return "\n".join(lines), vector_count, graph_count


query = "What nationality was Oliver Reed's character in the film Royal Flash?"
print(f"Query: {query}")
print(f"Gold answer: Prussian")
print()

for alpha in [0.0, 0.4, 0.6, 1.0]:
    ctx, v_count, g_count = assemble_context(
        query, collection, adj, alpha=alpha
    )
    print(f"\nalpha={alpha}: {v_count} vector chunks + {g_count} graph paths")
    # Show first 3 lines of context
    for line in ctx.split("\n")[:6]:
        if line.strip():
            print(f"  {line}")

Query: What nationality was Oliver Reed's character in the film Royal Flash?
Gold answer: Prussian


alpha=0.0: 4 vector chunks + 0 graph paths
  [Context | alpha=0.0]
  -- Vector (4 chunks, weight 1.0) --
    [sim=0.699] The character was played by actor Oliver Reed in the film of...
    [sim=0.503] [Oliver Reed] Robert Oliver Reed (13 February 1938 – 2 May 1...
    [sim=0.456] Notable films include "The Trap" (1966), "Oliver!...

alpha=0.4: 2 vector chunks + 4 graph paths
  [Context | alpha=0.4]
  -- Vector (2 chunks, weight 0.6) --
    [sim=0.699] The character was played by actor Oliver Reed in the film of...
    [sim=0.503] [Oliver Reed] Robert Oliver Reed (13 February 1938 – 2 May 1...

alpha=0.6: 1 vector chunks + 6 graph paths
  [Context | alpha=0.6]
  -- Vector (1 chunks, weight 0.4) --
    [sim=0.699] The character was played by actor Oliver Reed in the film of...
  -- Graph BFS (6 paths, weight 0.6) --

alpha=1.0: 1 vector chunks + 12 graph paths
  [Context | alpha=1.0]
  --

---
## Section 10 — What Do Competing Systems Actually Use?

This is one of the most important sections for understanding the benchmark. Before we claim our system is novel, we need to know exactly what the competition uses.

### Verified retrieval methods per system

All information below is from primary papers, not secondary summaries.

**Self-RAG** (Asai et al., ICLR 2024, arXiv:2310.11511)
- Retrieval: Dense vector only (DPR-style bi-encoder)
- No graph traversal, no BM25, no reranker
- Self-reflection operates at token level (special tokens `[Retrieve]`, `[IsRel]`, `[IsSup]`)
- Our comparison: we have graph traversal; they do not

**Microsoft GraphRAG** (Edge et al., 2024, arXiv:2404.16130)
- Retrieval: Graph traversal only — entity neighbourhood BFS for local queries, community summaries for global
- No vector similarity search in retrieval step, no BM25, no reranker
- Evaluated on community-level queries, not HotpotQA/MuSiQue

**LightRAG** (Guo et al., EMNLP 2025, arXiv:2410.05779)
- Retrieval: Dual-level graph retrieval — entity-level vector similarity + high-level keyword retrieval, both over the knowledge graph
- No traditional text chunk vector search (ChromaDB-style), no BM25, no cross-encoder reranker
- Evaluated on UltraDomain, not HotpotQA/MuSiQue

**HippoRAG 2** (Gutiérrez et al., ICML 2025, arXiv:2502.14802)
- Retrieval: Dense vector retrieval seeds Personalized PageRank (PPR) graph propagation
- Passage nodes embedded directly in the knowledge graph (closest to our design)
- No BM25, no cross-encoder reranker
- Uses BM25s library for BM25 as a comparison baseline only (not in their system)

**Graph-R1** (Luo et al., ICML 2026, arXiv:2507.21892)
- Retrieval: RL-trained agent executing multi-turn "think-retrieve-rethink" over a knowledge hypergraph
- No vector search, no BM25, no reranker — the RL policy learns what to retrieve
- Requires GPU for RL training; cannot run on our machine

**GraphRAG-R1** (Yu et al., WWW 2026, arXiv:2507.23581)
- Retrieval: "Hybrid graph-textual retrieval" — text chunk vector search + graph subgraph retrieval
- No BM25, no cross-encoder reranker
- RL-trained (GRPO) for adaptive retrieval invocation; requires GPU
- Their ablation: Text+Graph > Text only > Graph only — **validates our alpha-fusion design**

### The key conclusion

**None of the systems we compare against use BM25 or cross-encoder reranking on HotpotQA, MuSiQue, or 2WikiMultiHopQA.**

Our retrieval combination (ChromaDB dense + BFS graph, no BM25, no reranker) is directly comparable to every system in the table. We are not missing something the competition has — we are on a level playing field.

Adding BM25 would be an improvement **over all of them**, not just a catch-up move.

In [9]:
# Display the verified retrieval comparison table

systems = [
    {
        "name":      "SciGraphAgent (ours)",
        "dense":     "✓ ChromaDB 384-dim",
        "graph":     "✓ BFS co-occurrence",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✗ CPU only",
        "hotpotqa":  "TBD",
        "paper":     "This work",
    },
    {
        "name":      "Self-RAG",
        "dense":     "✓ DPR bi-encoder",
        "graph":     "✗",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✓ required",
        "hotpotqa":  "0.450",
        "paper":     "arXiv:2310.11511",
    },
    {
        "name":      "HippoRAG 2",
        "dense":     "✓ dense + PPR",
        "graph":     "✓ Personalised PR",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✓ required",
        "hotpotqa":  "~0.50",
        "paper":     "arXiv:2502.14802",
    },
    {
        "name":      "MS GraphRAG",
        "dense":     "✗",
        "graph":     "✓ community BFS",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✗",
        "hotpotqa":  "N/A",
        "paper":     "arXiv:2404.16130",
    },
    {
        "name":      "GraphRAG-R1",
        "dense":     "✓ text chunks",
        "graph":     "✓ subgraph",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✓ RL training",
        "hotpotqa":  "~0.62",
        "paper":     "arXiv:2507.23581",
    },
    {
        "name":      "Graph-R1",
        "dense":     "✗",
        "graph":     "✓ hypergraph RL",
        "bm25":      "✗",
        "reranker":  "✗",
        "gpu":       "✓ RL training",
        "hotpotqa":  "~0.58",
        "paper":     "arXiv:2507.21892",
    },
]

print(f"{'System':<22} {'Dense':>16} {'Graph':>18} {'BM25':>5} {'Rerank':>7} {'GPU':>11} {'HotpotQA F1':>12}")
print("-" * 95)
for s in systems:
    print(
        f"{s['name']:<22} "
        f"{s['dense']:>16} "
        f"{s['graph']:>18} "
        f"{s['bm25']:>5} "
        f"{s['reranker']:>7} "
        f"{s['gpu']:>11} "
        f"{s['hotpotqa']:>12}"
    )

print()
print("Key finding: NO system in this comparison uses BM25 or reranking")
print("on HotpotQA/MuSiQue/2WikiMultiHop. Our retrieval is directly comparable.")
print()
print("Our unique contribution is NOT the retrieval method —")
print("it is the RAGAS runtime evaluation gate + GPU-free + open-source combination.")

System                            Dense              Graph  BM25  Rerank         GPU  HotpotQA F1
-----------------------------------------------------------------------------------------------
SciGraphAgent (ours)   ✓ ChromaDB 384-dim ✓ BFS co-occurrence     ✗       ✗  ✗ CPU only          TBD
Self-RAG               ✓ DPR bi-encoder                  ✗     ✗       ✗  ✓ required        0.450
HippoRAG 2                ✓ dense + PPR  ✓ Personalised PR     ✗       ✗  ✓ required        ~0.50
MS GraphRAG                           ✗    ✓ community BFS     ✗       ✗           ✗          N/A
GraphRAG-R1               ✓ text chunks         ✓ subgraph     ✗       ✗ ✓ RL training        ~0.62
Graph-R1                              ✗    ✓ hypergraph RL     ✗       ✗ ✓ RL training        ~0.58

Key finding: NO system in this comparison uses BM25 or reranking
on HotpotQA/MuSiQue/2WikiMultiHop. Our retrieval is directly comparable.

Our unique contribution is NOT the retrieval method —
it is the RAGAS r

---
## Section 11 — The Four Retrieval Conditions

The ablation study (Experiment 2) tests four conditions to isolate the contribution of each retrieval signal.

### Why four conditions?

If we only compared Condition D (hybrid) against a single baseline, we could not tell whether the improvement came from the graph, the vector search, or their combination. By testing all four conditions on identical questions, we can attribute the performance difference.

| Condition | Method | Purpose |
|---|---|---|
| A | No retrieval | LLM baseline — what does parametric memory alone achieve? |
| B | Vector only (α=0.0) | Standard RAG baseline — industry standard |
| C | Graph only (α=1.0) | Pure graph baseline — structural context alone |
| D | Hybrid (α=0.6) | Our proposed system — does combining help? |

### The prediction

Based on Han et al. (2025):
- D should outperform B by ≥ +15 percentage points in context recall
- D should outperform C on single-hop questions (vector helps there)
- B should outperform A (any retrieval helps vs no retrieval)
- C may or may not outperform B — depends on graph quality

In [10]:
# Implement and demonstrate all four retrieval conditions

class ConditionA:
    """No retrieval — empty context, LLM uses parametric memory."""
    name = "A: No retrieval"
    def retrieve(self, query, top_k=4): return ""

class ConditionB:
    """Vector-only RAG — ChromaDB cosine similarity."""
    name = "B: Vector-only (alpha=0.0)"
    def __init__(self, col): self.col = col
    def retrieve(self, query, top_k=4):
        n = min(top_k, self.col.count())
        r = self.col.query(query_texts=[query], n_results=n,
                           include=["documents", "distances"])
        lines = ["[Vector Context]"]
        for doc, dist in zip(r["documents"][0], r["distances"][0]):
            lines.append(f"[sim={round(1-dist,3)}] {doc[:80]}")
        return "\n".join(lines)

class ConditionC:
    """Graph-only BFS — no vector search."""
    name = "C: Graph-only (alpha=1.0)"
    def __init__(self, adj): self.adj = adj
    def retrieve(self, query, top_k=4):
        words = re.findall(r'\b[a-z]{4,}\b', query.lower())
        seeds = [w for w in words if w in self.adj][:4]
        caps  = re.findall(r'\b([A-Z][a-z]+)\b', query)
        seeds += [c.lower() for c in caps if c.lower() in self.adj]
        seeds = list(dict.fromkeys(seeds))[:4]
        paths, visited = [], set()
        queue = [(s, [s], 0) for s in seeds]
        while queue and len(paths) < top_k * 3:
            curr, path, d = queue.pop(0)
            if curr in visited or d > 2: continue
            visited.add(curr)
            for nb, src in self.adj.get(curr, [])[:3]:
                if nb not in visited:
                    paths.append(" → ".join(path + [nb]))
                    queue.append((nb, path+[nb], d+1))
        lines = ["[Graph Context]", f"Seeds: {seeds}"]
        lines.extend(paths[:top_k*3])
        return "\n".join(lines)

class ConditionD:
    """Hybrid Graph-RAG — alpha-weighted fusion."""
    name = "D: Hybrid Graph-RAG (alpha=0.6)"
    def __init__(self, col, adj, alpha=0.6):
        self.col = col
        self.adj = adj
        self.alpha = alpha
    def retrieve(self, query, top_k=4):
        ctx, _, _ = assemble_context(query, self.col, self.adj, self.alpha, top_k)
        return ctx

# Instantiate all four
retrievers = [
    ConditionA(),
    ConditionB(collection),
    ConditionC(adj),
    ConditionD(collection, adj, alpha=0.6),
]

# Test on all three questions
print("=" * 65)
print("FOUR CONDITIONS — RETRIEVAL OUTPUT COMPARISON")
print("=" * 65)

for rec in records:
    print(f"\nQ: {rec['question']}")
    print(f"A: {rec['answer']}")
    print()
    for ret in retrievers:
        ctx = ret.retrieve(rec["question"], top_k=3)
        ctx_lines = [l for l in ctx.split("\n") if l.strip()]
        preview = ctx_lines[1] if len(ctx_lines) > 1 else "(empty)"
        print(f"  [{ret.name}]")
        print(f"    {preview[:90]}...")
        # Check if answer appears in context
        answer_found = rec['answer'].lower() in ctx.lower()
        print(f"    Answer '{rec['answer']}' in context: {answer_found}")
    print("-" * 65)

FOUR CONDITIONS — RETRIEVAL OUTPUT COMPARISON

Q: What nationality was Oliver Reed's character in the film Royal Flash?
A: Prussian

  [A: No retrieval]
    (empty)...
    Answer 'Prussian' in context: False
  [B: Vector-only (alpha=0.0)]
    [sim=0.699] The character was played by actor Oliver Reed in the film of the same name....
    Answer 'Prussian' in context: False
  [C: Graph-only (alpha=1.0)]
    Seeds: ['oliver', 'reed']...
    Answer 'Prussian' in context: False
  [D: Hybrid Graph-RAG (alpha=0.6)]
    -- Vector (1 chunks, weight 0.4) --...
    Answer 'Prussian' in context: False
-----------------------------------------------------------------

Q: Pacific Mozart Ensemble performed which German composer's Der Lindberghflug in 2002?
A: Kurt Julian Weill

  [A: No retrieval]
    (empty)...
    Answer 'Kurt Julian Weill' in context: False
  [B: Vector-only (alpha=0.0)]
    [sim=0.333] Overall, it was the third production the Berliner Ensemble performed....
    Answer 'Kurt Julian

---
## Section 12 — Running `step02_build_retrieval_systems.py` — Line by Line

Now we run the actual script to understand each component in context.

In [11]:
# Run step02 directly from the notebook
import importlib.util
from pathlib import Path

script_path = Path("step02_build_retrieval_systems.py")
if not script_path.exists():
    script_path = Path("..") / "step02_build_retrieval_systems.py"

print(f"Loading script from: {script_path.resolve()}")
spec = importlib.util.spec_from_file_location("step02", script_path)
step02 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(step02)

print()
print("Running step02.main(dataset='hotpotqa', n=3)...")
print()
result = step02.main(dataset="hotpotqa", n=3)

Loading script from: /run/media/bala/HDD/Projects/scigraphagent-benchmark/step02_build_retrieval_systems.py

Running step02.main(dataset='hotpotqa', n=3)...


  STEP 02: BUILD RETRIEVAL SYSTEMS
  Dataset  : hotpotqa
  N        : 3
  Index dir: /run/media/bala/HDD/Projects/scigraphagent-benchmark/index

  Loaded 3 records from hotpotqa_sample_3.json

  [A] Building ChromaDB vector index...
     Indexed  : 36 chunks from 3 records
     Time     : 2.6s
     Model    : all-MiniLM-L6-v2 (384-dim, CPU)
     Storage  : index/chroma

  [B] Building in-memory knowledge graph...
     Nodes    : 411
     Edges    : 426
     Density  : 1.04 edges/node
     Time     : 0.00s

───────────────────────────────────────────────────────
  SMOKE TEST — Retrieval preview on question 1
───────────────────────────────────────────────────────
  Question : What nationality was Oliver Reed's character in the film Royal Flash?
  Answer   : Prussian


  Vector retrieval demo — query: 'What nationality was Oliver R

In [12]:
# Inspect the saved config file
import json
from pathlib import Path

config_path = Path("data") / "retrieval_config.json"
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    print("retrieval_config.json:")
    print(json.dumps(config, indent=2))
    print()
    print("This file tells step03 which collection and alpha to use.")
    print(f"  Knowledge graph: {config['kg_stats']['nodes']} nodes, {config['kg_stats']['edges']} edges")
    print(f"  Alpha (graph weight): {config['alpha']}")
else:
    print("Config file not found — run Section 12 first.")

retrieval_config.json:
{
  "dataset": "hotpotqa",
  "n_records": 3,
  "collection_name": "benchmark_hotpotqa_3",
  "alpha": 0.6,
  "kg_stats": {
    "nodes": 411,
    "edges": 426,
    "density": 1.04,
    "avg_degree": 2.07
  },
  "retrievers": [
    {
      "id": "A",
      "name": "A: No retrieval (LLM only)",
      "alpha": 0.0
    },
    {
      "id": "B",
      "name": "B: Vector-only RAG (alpha=0.0)",
      "alpha": 0.0
    },
    {
      "id": "C",
      "name": "C: Graph-only BFS (alpha=1.0)",
      "alpha": 1.0
    },
    {
      "id": "D",
      "name": "D: Hybrid Graph-RAG (alpha=0.6)",
      "alpha": 0.6
    }
  ]
}

This file tells step03 which collection and alpha to use.
  Knowledge graph: 411 nodes, 426 edges
  Alpha (graph weight): 0.6


---
## Section 13 — Pre-Commit Verification and Commit

Before committing, verify all outputs are correct and the script is ready.

In [13]:
import json, subprocess
from pathlib import Path

print("=" * 55)
print("PRE-COMMIT VERIFICATION")
print("=" * 55)

checks = []
def check(desc, ok, detail=""):
    checks.append(ok)
    s = "✓" if ok else "✗"
    print(f"  {s}  {desc}")
    if detail: print(f"       {detail}")

# Script exists
check("step02_build_retrieval_systems.py exists",
      Path("step02_build_retrieval_systems.py").exists())

# ChromaDB index exists
chroma_path = Path("index") / "chroma"
check("ChromaDB index created", chroma_path.exists(),
      str(chroma_path.resolve()))

# Config file saved
config_path = Path("data") / "retrieval_config.json"
if config_path.exists():
    with open(config_path) as f:
        cfg = json.load(f)
    check("retrieval_config.json saved", True,
          f"kg: {cfg['kg_stats']['nodes']} nodes, {cfg['kg_stats']['edges']} edges")
    check("Alpha is 0.6", cfg["alpha"] == 0.6)
    check("4 retrievers defined", len(cfg["retrievers"]) == 4)
else:
    check("retrieval_config.json saved", False)

# Verify all four retriever conditions are in config
if config_path.exists():
    cond_ids = [r["id"] for r in cfg["retrievers"]]
    check("Condition A present", "A" in cond_ids)
    check("Condition B present", "B" in cond_ids)
    check("Condition C present", "C" in cond_ids)
    check("Condition D present", "D" in cond_ids)

# Git status
print()
print("Git status:")
result = subprocess.run("git status --short", shell=True,
                        capture_output=True, text=True)
for line in result.stdout.strip().split("\n"):
    if line.strip():
        print(f"  {line}")

print()
all_ok = all(checks)
if all_ok:
    print("✓ All checks passed — ready to commit")
else:
    print(f"✗ {sum(1 for c in checks if not c)} checks failed")

print()
print("Run in terminal:")
print("  git add step02_build_retrieval_systems.py notebooks/02_build_retrieval_systems.ipynb")
print('  git commit -m "Add step02 and Notebook 02: vector index and knowledge graph builder"')
print("  git push")

PRE-COMMIT VERIFICATION
  ✓  step02_build_retrieval_systems.py exists
  ✓  ChromaDB index created
       /run/media/bala/HDD/Projects/scigraphagent-benchmark/index/chroma
  ✓  retrieval_config.json saved
       kg: 411 nodes, 426 edges
  ✓  Alpha is 0.6
  ✓  4 retrievers defined
  ✓  Condition A present
  ✓  Condition B present
  ✓  Condition C present
  ✓  Condition D present

Git status:
  ?? 02_build_retrieval_systems.ipynb
  ?? step02_build_retrieval_systems.py

✓ All checks passed — ready to commit

Run in terminal:
  git add step02_build_retrieval_systems.py notebooks/02_build_retrieval_systems.ipynb
  git commit -m "Add step02 and Notebook 02: vector index and knowledge graph builder"
  git push


---
## Summary — What We Learned

### Concepts covered

| Concept | What it is | Where it appears |
|---|---|---|
| Embedding | Text → 384-dim float vector capturing meaning | all-MiniLM-L6-v2 |
| Cosine similarity | Angle between vectors → 0 to 1 similarity score | ChromaDB retrieval |
| Vector database | ANN index for fast nearest-neighbour search | ChromaDB |
| HNSW | Multi-layer graph for O(log n) ANN search | ChromaDB internals |
| Knowledge graph | Nodes (entities) + edges (relations) | KnowledgeGraph class |
| Co-occurrence | Two entities in same passage → edge | Entity extraction |
| BFS | Level-by-level graph traversal | Graph retrieval |
| Alpha fusion | α × graph + (1-α) × vector = hybrid context | Condition D |
| Complementarity | Vector strong on single-hop, graph on multi-hop | Han et al. 2025 |
| Adaptive alpha | Planned: classify query type, adjust α | Improvement roadmap |

### Files created by this step

```
index/chroma/          ← ChromaDB persistent vector store
data/retrieval_config.json ← config for step03
```

### What comes next — Notebook 03

**Notebook 03 — Running Experiments**

We use the four retrieval conditions to run three experiments:
1. Does the RAGAS retry gate improve faithfulness? (primary novelty)
2. Does hybrid Graph-RAG outperform vector-only? (ablation)
3. Does the CI/CD gate catch regressions? (engineering novelty)

Every API call goes to Groq (free tier). The RAGAS retry gate — the central novelty of SciGraphAgent — fires for the first time in Experiment 1.

---
*Notebook 02 complete. Commit and push, then continue with `03_run_experiments.ipynb`.*